# SPARK 2026 | W9-CXR: Chest X-Ray Pneumonia Classification with Transfer Learning

## Imports & Paths

In [1]:
# Imports
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from tqdm import tqdm
import torchvision

# Dataset path
BASE_PATH = "/kaggle/input/datasets/mouhamed221/dataset-spark-week7/dataset/"

print(BASE_PATH)

/kaggle/input/datasets/mouhamed221/dataset-spark-week7/dataset/


## Transforms & Dataset

In [3]:
# Data augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Validation/Test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Custom Dataset
class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, base_path, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.base_path = base_path
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.base_path + self.df.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, self.df.iloc[idx]['id']
        else:
            label = self.df.iloc[idx]['target']
            return image, torch.tensor(label, dtype=torch.float32)

# Dataset instances
train_dataset = ChestXrayDataset(BASE_PATH + "train.csv", BASE_PATH, transform=train_transform)
val_dataset   = ChestXrayDataset(BASE_PATH + "val.csv", BASE_PATH, transform=val_transform)
test_dataset  = ChestXrayDataset(BASE_PATH + "test.csv", BASE_PATH, transform=val_transform, is_test=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("-------------------Succes !--------------------")

-------------------Succes !--------------------


In [4]:
# Pos_weight helps the model focus more on the minority class (pneumonia),
pos_weight = torch.tensor([
    (len(train_dataset) - train_dataset.df['target'].sum()) 
    / train_dataset.df['target'].sum()
]).to(device)

print("-------------------Succes !--------------------")

-------------------Succes !--------------------


## Model Transfer Learning :
### The model is pretrained on ImageNet, which provides strong low-level features
### such as edges and textures. However, medical images differ significantly
### from natural images, so fine-tuning is often necessary.

In [6]:
def build_model(mode="frozen"):
    """
    mode:
    - 'frozen'      -> feature extraction
    - 'finetune'    -> unfreeze last block (layer4)
    - 'full'        -> train entire network
    """

    model = torchvision.models.resnet18(weights='IMAGENET1K_V1')

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    if mode == "finetune":
        # Unfreeze last convolutional block
        for param in model.layer4.parameters():
            param.requires_grad = True

    elif mode == "full":
        # Unfreeze entire model
        for param in model.parameters():
            param.requires_grad = True

    # Replace final layer for binary classification
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 1)

    return model.to(device)


print("----------------------Succes !---------------------")

----------------------Succes !---------------------


## Training :
### Frozen model learns only the classifier → fast but limited adaptation
### Fine-tuning allows domain adaptation → better performance
### Full training may overfit due to limited dataset size

In [8]:
def train_model(model, epochs=10):
    # Loss function for binary classification
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


    # Optimize only trainable parameters
    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,
        momentum=0.9
    )

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in tqdm(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_preds, val_labels, val_probs = [], [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images).squeeze()
                probs = torch.sigmoid(outputs)

                val_probs.extend(probs.cpu().tolist())
                val_preds.extend((probs > 0.5).int().cpu().tolist())
                val_labels.extend(labels.int().cpu().tolist())

        # Metrics
        accuracy = (torch.tensor(val_preds) == torch.tensor(val_labels)).float().mean()
        f1 = f1_score(val_labels, val_preds)

        try:
            auc = roc_auc_score(val_labels, val_probs)
        except:
            auc = 0.0

        print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.4f} | Acc: {accuracy:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
        cm = confusion_matrix(val_labels, val_preds)
        print("Confusion Matrix:")
        print(cm)

    return accuracy, f1, auc

print("----------------------Succes !---------------------")

----------------------Succes !---------------------


## Experiment Design:
### We compare three strategies:
### 1. Frozen (feature extraction)
### 2. Fine-tuning (partial adaptation)
### 3. Full training (maximum flexibility but risk of overfitting)

In [9]:
results = {}

# 1. Frozen model (feature extraction)
print("=== Frozen Model ===")
model_frozen = build_model("frozen")
results["frozen"] = train_model(model_frozen, epochs=10)

# 2. Fine-tuning (partial)
print("=== Fine-tuning (layer4) ===")
model_ft = build_model("finetune")
results["finetune"] = train_model(model_ft, epochs=10)

# 3. Full training
print("=== Full Training ===")
model_full = build_model("full")
results["full"] = train_model(model_full, epochs=10)

=== Frozen Model ===
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s] 
100%|██████████| 129/129 [01:40<00:00,  1.28it/s]


Epoch 1/10 - Loss: 0.3652 | Acc: 0.6310 | F1: 0.7055 | AUC: 0.7269
Confusion Matrix:
[[166  71]
 [253 388]]


100%|██████████| 129/129 [01:17<00:00,  1.67it/s]


Epoch 2/10 - Loss: 0.3115 | Acc: 0.7825 | F1: 0.8346 | AUC: 0.8905
Confusion Matrix:
[[205  32]
 [159 482]]


...

Epoch 8/10 - Loss: 0.1101 | Acc: 0.9146 | F1: 0.9386 | AUC: 0.9809
Confusion Matrix:
[[230   7]
 [ 68 573]]


100%|██████████| 129/129 [01:18<00:00,  1.65it/s]


Epoch 9/10 - Loss: 0.1050 | Acc: 0.9305 | F1: 0.9506 | AUC: 0.9823
Confusion Matrix:
[[230   7]
 [ 54 587]]


100%|██████████| 129/129 [01:22<00:00,  1.56it/s]


Epoch 10/10 - Loss: 0.1003 | Acc: 0.9442 | F1: 0.9611 | AUC: 0.9830
Confusion Matrix:
[[224  13]
 [ 36 605]]


## Final Analysis:
### - Week 7 CNN performs very well despite random initialization
### - Transfer learning converges faster and is more stable
### - Fine-tuning usually provides the best trade-off between performance and generalization
### - Full training can overfit if the dataset is small

In [11]:
print("\n=== Comparison with Week 7 CNN ===")

print("Week 7 CNN (from scratch):")
print("Acc: 0.946 | F1: 0.963 | AUC: 0.989\n")

for k, v in results.items():
    print(f"{k} -> Acc: {v[0]:.4f} | F1: {v[1]:.4f} | AUC: {v[2]:.4f}")

# Select best model based on AUC
my_best_model = max(results, key=lambda x: results[x][2])
print("\nBest model based on AUC:", my_best_model)


=== Comparison with Week 7 CNN ===
Week 7 CNN (from scratch):
Acc: 0.946 | F1: 0.963 | AUC: 0.989

frozen -> Acc: 0.8531 | F1: 0.8889 | AUC: 0.9641
finetune -> Acc: 0.8907 | F1: 0.9197 | AUC: 0.9794
full -> Acc: 0.9442 | F1: 0.9611 | AUC: 0.9830

Best model based on AUC: full


## Prediction and submission file

In [12]:
# choose best model
if my_best_model == "frozen":
    model = model_frozen
elif my_best_model == "finetune":
    model = model_ft
else:
    model = model_full

model.eval()
predictions, ids = [], []

with torch.no_grad():
    for images, img_ids in test_loader:
        images = images.to(device)

        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()

        predictions.extend(preds.cpu().tolist())
        ids.extend(img_ids)

submission = pd.DataFrame({'id': ids, 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print("Submission file saved!")

Submission file saved!
